# Cleaning: health + metadata

Urutan: **0 Setup → 1 Load data → 2 metadata_dates → 3 Health → 4 Metadata → 5 Simpan**. Sel 5 adalah sel terakhir.

## 0. Setup

In [1]:
import os, re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

DATA = "../data"

HEALTH_COUNTS = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases",
                 "copd_admissions", "resp", "cvd", "death"]
HEALTH_WEATHER = ["temp_c", "dptp_c", "rhum_pct"]
EXPECTED_ROWS = 24432            # 1 Jan 2024 00:00 s.d. 14 Okt 2026 23:00, per jam
SENTINELS = [-9999, -999, -99.9, 9999, 99999]

cleaning_log = []
def log(step, n, note=""):
    cleaning_log.append({"step": step, "rows_affected": int(n), "note": note})
    print(f"[LOG] {step}: {int(n):,} {note}")

def city_key(s):
    """Kunci join wilayah. Awalan Kota/Kabupaten TIDAK dihapus (Kota Bogor != Kabupaten Bogor)."""
    return (s.astype(str).str.replace("Kota Administrasi ", "", regex=False)
             .str.strip().str.replace(r"\s+", " ", regex=True).str.lower())


## 1. Load data


In [3]:
from pathlib import Path

SEARCH_ROOT = Path("..").resolve()

def find_file(name, max_depth=4):
    hits = []
    for dirpath, dirnames, filenames in os.walk(SEARCH_ROOT):
        depth = len(Path(dirpath).relative_to(SEARCH_ROOT).parts)
        dirnames[:] = [d for d in dirnames
                       if not d.startswith(".") and d not in ("node_modules", "__pycache__", "clean")
                       and depth < max_depth]
        if name in filenames:
            hits.append(Path(dirpath) / name)
    if not hits:
        raise FileNotFoundError(f"'{name}' tidak ditemukan di bawah {SEARCH_ROOT}. "
                                f"Letakkan file itu di folder proyek atau ubah SEARCH_ROOT.")
    if len(hits) > 1:
        print(f"[info] {name} ada {len(hits)} salinan, memakai: {hits[0]}")
    return hits[0]

FILES = {"sites": "sites.csv", "sensors": "sensors.csv", "hospitals": "hospitals.csv",
         "health": "HLT-01_health_outcomes_daily.csv",
         "m_dates": "metadata_dates.csv", "m_sensors": "metadata_sensors.csv"}

sites     = pd.read_csv(find_file(FILES["sites"]))
sensors   = pd.read_csv(find_file(FILES["sensors"]))
hospitals = pd.read_csv(find_file(FILES["hospitals"]))
health    = pd.read_csv(find_file(FILES["health"]))
m_dates   = pd.read_csv(find_file(FILES["m_dates"]))
m_sensors = pd.read_csv(find_file(FILES["m_sensors"]))

for n, d in {"sites": sites, "sensors": sensors, "hospitals": hospitals,
             "health": health, "m_dates": m_dates, "m_sensors": m_sensors}.items():
    print(f"{n:10s} {d.shape}")
print("\nHasil cleaning akan disimpan ke:", Path(DATA).resolve() / "clean")


sites      (27, 27)
sensors    (27, 10)
hospitals  (27, 7)
health     (27491, 14)
m_dates    (27, 6)
m_sensors  (16, 6)

Hasil cleaning akan disimpan ke: C:\Users\ASUS\Downloads\Gelar Rasa 2026\dagelarrasa\data\clean


## 2. metadata_dates + START/END

In [4]:
m_dates["date_start"] = pd.to_datetime(m_dates["date_start"], errors="raise")
m_dates["date_end"] = pd.to_datetime(m_dates["date_end"], errors="raise")

assert m_dates["location_id"].is_unique, "location_id duplikat di metadata_dates"
assert m_dates[["date_start", "date_end"]].notna().all().all()
assert (m_dates["date_end"] > m_dates["date_start"]).all()
assert m_dates["date_start"].nunique() == 1 and m_dates["date_end"].nunique() == 1

START = m_dates["date_start"].min()      # 2024-01-01 00:00
END = m_dates["date_end"].max()          # 2026-10-14 23:00
print("Periode metadata:", START, "->", END)


Periode metadata: 2024-01-01 00:00:00 -> 2026-10-14 23:00:00


## 3. Cleaning health (HLT-01)
Harus berakhir dengan log `health bersih (hasil): 27,486`.

In [7]:
# Hasil: DataFrame `health_clean` (data mentah `health` tidak diubah).
health_clean = health.copy()
health_clean.columns = health_clean.columns.str.strip().str.lower()
log("health awal", len(health_clean))

# (1) Tipe data: teks di kolom angka -> NaN, id & tanggal dirapikan
health_clean["hospital_id"] = health_clean["hospital_id"].astype("string").str.strip().str.upper()
health_clean["date"] = pd.to_datetime(health_clean["date"], format="%Y-%m-%d", errors="raise")
for c in HEALTH_COUNTS + HEALTH_WEATHER:
    conv = pd.to_numeric(health_clean[c], errors="coerce")
    n_text = int((conv.isna() & health_clean[c].notna()).sum())
    health_clean[c] = conv
    if n_text:
        log(f"{c}: teks error -> NaN", n_text)

# (2) Kode penanda (sentinel) -> NaN
for c in HEALTH_COUNTS + HEALTH_WEATHER:
    m = health_clean[c].isin(SENTINELS)
    if m.any():
        health_clean.loc[m, c] = np.nan
        log(f"{c}: kode penanda -> NaN", m.sum())

# (3) Di luar batas masuk akal -> NaN
HEALTH_LIMITS = {**{c: (0, 1000) for c in HEALTH_COUNTS},
                 "temp_c": (-10, 50), "dptp_c": (-10, 50), "rhum_pct": (0, 100)}
for c, (lo, hi) in HEALTH_LIMITS.items():
    m = health_clean[c].notna() & ((health_clean[c] < lo) | (health_clean[c] > hi))
    if m.any():
        health_clean.loc[m, c] = np.nan
        log(f"{c}: di luar [{lo}, {hi}] -> NaN", m.sum())

# (4) Duplikat (hospital_id + date)
n_dup = int(health_clean.duplicated(["hospital_id", "date"]).sum())
health_clean = health_clean.drop_duplicates(["hospital_id", "date"], keep="first")
log("duplikat hospital_id+date dibuang", n_dup)

# (5) Kolom dow dihitung ulang dari date
n_dow = int((health_clean["date"].dt.day_name() != health_clean["dow"]).sum())
health_clean["dow"] = health_clean["date"].dt.day_name()
log("dow tidak cocok dengan date (diperbaiki)", n_dow)

# (6) Kolom `time`: BUKAN jam, melainkan nomor urut hari per RS dan tidak sinkron antar RS
#     (RS 12, 13, 16, 19, 20 mulai 1 di 2023-12-31; RS lain mulai 1 di 2024-01-01).
#     Dibuang, diganti day_index yang dihitung dari tanggal.
health_clean = health_clean.drop(columns=["time"])

# (7) Baris di luar periode metadata (2024-01-01 s.d. 2026-10-14).
#     5 baris 2023-12-31 (RS 12, 13, 16, 19, 20): stasiun AQMS-nya baru mulai 23:55, jadi hanya
#     mewakili 1 jam data (pipeline AQMS membulatkan ke jam 2024-01-01 00:00) -> dibuang.
out_window = ~health_clean["date"].between(START.normalize(), END.normalize())
print(health_clean.loc[out_window, ["date", "hospital_id"]].to_string())
health_clean = health_clean.loc[~out_window].copy()
log("baris di luar periode metadata dibuang", out_window.sum())
health_clean["day_index"] = (health_clean["date"] - START.normalize()).dt.days + 1

# (8) Tipe akhir: hitungan = integer (sekaligus menghilangkan nilai "-0.0")
for c in HEALTH_COUNTS:
    assert health_clean[c].notna().all(), f"{c} masih ada NaN -> cek log di atas"
    assert ((health_clean[c] % 1) == 0).all(), f"{c} berisi pecahan"
    health_clean[c] = health_clean[c].astype("Int64")

# (9) Validasi
assert health_clean[HEALTH_WEATHER].notna().all().all()
assert (health_clean["dptp_c"] <= health_clean["temp_c"]).all(), "titik embun > suhu"
expected = pd.date_range(START.normalize(), END.normalize(), freq="D")
full = pd.MultiIndex.from_product([sorted(health_clean["hospital_id"].unique()), expected],
                                  names=["hospital_id", "date"])
lubang = full.difference(health_clean.set_index(["hospital_id", "date"]).index)
assert len(lubang) == 0, f"{len(lubang)} kombinasi RS-tanggal hilang"
assert set(health_clean["hospital_id"]) == set(hospitals["hospital_id"]), "hospital_id tidak cocok dgn hospitals.csv"

# (10) Flag (TIDAK dihapus)
#   - Lonjakan besar (RS-20 resp 307, RS-04/05 Sep-Okt 2026) naik kompak di semua kolom respirasi
#     pada hari berurutan -> pola episode kabut asap, bukan error. Dipertahankan.
#   - RS-27 punya 92 hari dengan resp < under5 + over60 (tidak konsisten) -> hanya ditandai.
health_clean["flag_resp_lt_age_sum"] = health_clean["resp"] < (
    health_clean["resp_admissions_under5"] + health_clean["resp_admissions_over60"])
log("ditandai resp < under5 + over60", health_clean["flag_resp_lt_age_sum"].sum(), "tidak dihapus")

health_clean = (health_clean[["date", "day_index", "dow", "hospital_id", *HEALTH_WEATHER,
                              *HEALTH_COUNTS, "flag_resp_lt_age_sum"]]
                .sort_values(["hospital_id", "date"]).reset_index(drop=True))
log("health bersih (hasil)", len(health_clean), "harus 27 RS x 1018 hari = 27.486")
assert len(health_clean) == 27 * 1018
display(health_clean.head())


[LOG] health awal: 27,491 
[LOG] duplikat hospital_id+date dibuang: 0 
[LOG] dow tidak cocok dengan date (diperbaiki): 0 
        date hospital_id
0 2023-12-31   RS-XXX-12
1 2023-12-31   RS-XXX-13
2 2023-12-31   RS-XXX-16
3 2023-12-31   RS-XXX-19
4 2023-12-31   RS-XXX-20
[LOG] baris di luar periode metadata dibuang: 5 
[LOG] ditandai resp < under5 + over60: 92 tidak dihapus
[LOG] health bersih (hasil): 27,486 harus 27 RS x 1018 hari = 27.486


,date,day_index,dow,hospital_id,temp_c,dptp_c,rhum_pct,resp_admissions_under5,resp_admissions_over60,asthma_emergency_cases,copd_admissions,resp,cvd,death,flag_resp_lt_age_sum
0,2024-01-01,1,Monday,RS-XXX-01,28.4,23.8,76.1,12,25,12,5,59,8,4,False
1,2024-01-02,2,Tuesday,RS-XXX-01,28.7,23.9,75.0,13,22,6,6,60,13,1,False
2,2024-01-03,3,Wednesday,RS-XXX-01,28.8,24.0,75.8,20,20,3,4,57,11,8,False
3,2024-01-04,4,Thursday,RS-XXX-01,28.5,23.8,75.7,15,32,12,3,81,8,3,False
4,2024-01-05,5,Friday,RS-XXX-01,28.7,24.0,76.0,16,21,7,6,58,11,3,False


## 4. Cleaning metadata (dim_location + sensor_headers)

In [8]:
import re

# --- (a) dim_location: jembatan hospital_id <-> location_id <-> sensor_id (untuk integrasi) ---
# city di metadata_dates tidak konsisten (LOC-STG-01 berisi nama provinsi "Kalimantan Barat"),
# jadi nama wilayah diambil dari sites.csv. hospitals <-> sites dijodohkan lewat city_key().
sites_k = sites.assign(_key=city_key(sites["kota_kabupaten"]))
hosp_k = hospitals.assign(_key=city_key(hospitals["kota_kabupaten"]))
assert sites_k["_key"].is_unique and hosp_k["_key"].is_unique

dim_location = (
    m_dates.drop(columns=["city"])
    .merge(sites_k[["location_id", "_key", "station_code", "provinsi", "kota_kabupaten", "kecamatan",
                    "region_group", "latitude_dd", "longitude_dd"]], on="location_id", how="left", validate="1:1")
    .merge(sensors[["sensor_id", "location_id", "nama_sensor", "model_sensor",
                    "status_operasional", "tanggal_instalasi"]], on="location_id", how="left", validate="1:1")
    .merge(hosp_k[["_key", "hospital_id", "nama_rumah_sakit", "kelas_faskes", "kapasitas_tempat_tidur"]],
           on="_key", how="left", validate="1:1")
    .drop(columns="_key")
)
# info tambahan dari kolom teks / angka
dim_location["resolution_hours"] = dim_location["nominal_resolution"].str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
dim_location["is_remote_station"] = dim_location["notes"].str.contains("remote", case=False)
dim_location["expected_days"] = (END.normalize() - START.normalize()).days + 1
dim_location["expected_hourly_rows"] = int((END - START) / pd.Timedelta(hours=1)) + 1   # = EXPECTED_ROWS (24432)
dim_location["kapasitas_tempat_tidur"] = dim_location["kapasitas_tempat_tidur"].astype("Int64")

assert len(dim_location) == 27
assert dim_location[["hospital_id", "sensor_id"]].notna().all().all(), "ada lokasi tanpa pasangan RS/sensor"
assert dim_location["hospital_id"].is_unique and dim_location["sensor_id"].is_unique
# sanity check: RS-XXX-NN selalu berpasangan dengan SENS-0NN
assert (dim_location["hospital_id"].str[-2:].astype(int) == dim_location["sensor_id"].str[-3:].astype(int)).all()
assert (dim_location["expected_hourly_rows"] == EXPECTED_ROWS).all()
log("dim_location dibangun", len(dim_location), "1 baris per stasiun")

# --- (b) sensor_headers: kolom Header di metadata_sensors berisi beberapa nama kolom sekaligus ---
ms = m_sensors.copy()
ms.columns = ms.columns.str.strip().str.lower()
for c in ms.columns:
    ms[c] = ms[c].astype("string").str.strip()
ms["header"] = ms["header"].str.split(r"\s+or\s+|\s*,\s*", regex=True)
sensor_headers = ms.explode("header").reset_index(drop=True)
sensor_headers["header"] = sensor_headers["header"].str.strip()

UNIT_RULES = [(r"_ug_m3$", "ug/m3"), (r"_ppm$", "ppm"), (r"^temperature_c$", "degC"),
              (r"^humidity_pct$", "%"), (r"^wind_speed_ms$", "m/s"),
              (r"^wind_direction_deg$", "deg"), (r"^qc_flag$", "categorical")]

def infer_unit(h):
    for pat, unit in UNIT_RULES:
        if re.search(pat, h):
            return unit
    raise ValueError(f"unit tidak dikenali: {h}")

sensor_headers["unit"] = sensor_headers["header"].map(infer_unit)     # menggantikan kolom units yang ambigu
sensor_headers["is_logger_export_unit"] = sensor_headers["unit"].eq("ppm")
sensor_headers = (sensor_headers.drop(columns=["units"])
                  .rename(columns={"parameters": "parameter", "sensors": "sensor_model", "type": "sensor_type"}))
assert not sensor_headers.duplicated(["supplier", "sensor_model", "parameter", "header"]).any()
log("sensor_headers dibangun", len(sensor_headers), "1 baris per (sensor, nama kolom)")
display(sensor_headers[["supplier", "sensor_model", "parameter", "header", "unit"]])


[LOG] dim_location dibangun: 27 1 baris per stasiun
[LOG] sensor_headers dibangun: 27 1 baris per (sensor, nama kolom)


,supplier,sensor_model,parameter,header,unit
0,Thermo Scientific,5014i BAM,Particulate Matter PM2.5,pm25_ug_m3,ug/m3
1,Thermo Scientific,5014i BAM,Particulate Matter PM2.5,pm25_ppm,ppm
2,Thermo Scientific,5014i BAM,Particulate Matter PM10,pm10_ug_m3,ug/m3
3,Thermo Scientific,5014i BAM,Particulate Matter PM10,pm10_ppm,ppm
4,Met One Instruments,BAM-1020,Particulate Matter PM2.5,pm25_ug_m3,ug/m3
5,Met One Instruments,BAM-1020,Particulate Matter PM2.5,pm25_ppm,ppm
6,Met One Instruments,BAM-1020,Particulate Matter PM10,pm10_ug_m3,ug/m3
7,Met One Instruments,BAM-1020,Particulate Matter PM10,pm10_ppm,ppm
8,Horiba,APDA-372,Particulate Matter PM2.5,pm25_ug_m3,ug/m3
9,Horiba,APDA-372,Particulate Matter PM10,pm10_ug_m3,ug/m3


## 5. Simpan hasil (sel terakhir)

In [9]:
for sub, frames in {"health": {"health_clean.csv": health_clean},
                    "metadata": {"dim_location.csv": dim_location, "sensor_headers.csv": sensor_headers}}.items():
    out_dir = f"{DATA}/clean/{sub}"
    os.makedirs(out_dir, exist_ok=True)
    for fname, df_out in frames.items():
        df_out.to_csv(os.path.join(out_dir, fname), index=False)
        print(f"Tersimpan: {out_dir}/{fname} | {len(df_out):,} baris, {df_out.shape[1]} kolom")


Tersimpan: ../data/clean/health/health_clean.csv | 27,486 baris, 15 kolom
Tersimpan: ../data/clean/metadata/dim_location.csv | 27 baris, 25 kolom
Tersimpan: ../data/clean/metadata/sensor_headers.csv | 27 baris, 7 kolom
